# Complete Coordinated ML Pipeline

This notebook provides a complete end-to-end machine learning pipeline that coordinates clean data preprocessing with XGBoost model training and evaluation using efficient count-based missingness tracking.

## Pipeline Overview
1. **Configuration**: Set all parameters in one place 
2. **Clean Data Preprocessing**: Streamlined feature engineering with count-based missingness
3. **XGBoost Model Training**: Hyperparameter optimization and model training with uncertainty quantification
4. **Pipeline Summary**: Display final metrics and save all artifacts

## Key Features
- **Count-Based Missingness**: Uses feature counts instead of redundant binary indicators
- **Streamlined Processing**: Efficient, maintainable preprocessing pipeline  
- **Dual Datasets**: Optimized for both XGBoost (preserves NaNs) and Elastic Net models (scaled & imputed)
- **100+ Count Features**: Track measurement frequency for all feature types
- **Hyperparameter Optimization**: Optuna-based automatic hyperparameter tuning
- **Uncertainty Quantification**: Bootstrap confidence intervals for robust evaluation
- **Complete Artifact Management**: Saves models, results, and logs
- **Error Handling**: Graceful handling of XGBoost installation issues


In [1]:
import os
import sys
import time
import pickle
import pandas as pd
import numpy as np
from datetime import datetime

# =============================================================================
# SHARED CONFIGURATION PARAMETERS
# =============================================================================

# -- File & Path Configuration --
HDF_FILE_PATH = '../../data/raw/all_hourly_data.h5'
FEATURE_CLASSIFICATION_PATH = '../../data/processed/eda_results_corrected/feature_classification.csv'
OUTPUT_DIR = 'phase_1_outputs'

# -- Experiment Configuration --
DRY_RUN = False  # Set to False for full dataset
DRY_RUN_PATIENTS = 1000  # Number of patients for dry run
USE_CACHED_PREPROCESSING = True  # Use cache if available

# -- Feature Engineering Options --
CALCULATE_TRENDS = True  # Include trend calculations
WINDOW_SIZE = 24  # Hours of data for feature extraction
GAP_TIME = 6      # Hours of gap before prediction

# -- Target and Reproducibility --
TARGET_VARIABLE = 'mort_hosp'
SEED = 42

# -- Hyperparameter Tuning Configuration --
N_OPTUNA_TRIALS = 15  # Number of trials for hyperparameter search
OPTUNA_TIMEOUT = 1800 # Timeout in seconds

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Clean Data Preprocessing Pipeline

Run the clean data preprocessing script with efficient count-based missingness tracking:


In [2]:
print("=" * 70)
print("PHASE 1: CLEAN DATA PREPROCESSING")
print("=" * 70)

preprocessing_start = time.time()

# Import and run preprocessing
try:
    # Import the clean preprocessing script as a module
    import importlib.util
    spec = importlib.util.spec_from_file_location("data_preprocessing_clean", "data_preprocessing_clean.py")
    if spec is None:
        raise ImportError("Could not load data_preprocessing_clean.py")
    
    preprocessing_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for data_preprocessing_clean.py")
    
    # Execute the preprocessing script
    print(f"Starting clean data preprocessing at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    spec.loader.exec_module(preprocessing_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'HDF_FILE_PATH': HDF_FILE_PATH,
        'FEATURE_CLASSIFICATION_PATH': FEATURE_CLASSIFICATION_PATH,
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'USE_CACHED_PREPROCESSING': USE_CACHED_PREPROCESSING,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED
    }
    
    # Call the main function explicitly with configuration
    preprocessing_module.main(config)
    
    preprocessing_time = time.time() - preprocessing_start
    print(f"\n✓ Clean data preprocessing completed in {preprocessing_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during preprocessing: {e}")
    raise


PHASE 1: CLEAN DATA PREPROCESSING
Starting clean data preprocessing at 2025-07-02 19:38:28


2025-07-02 19:38:48,344 [INFO] - Loading data...
2025-07-02 19:38:49,817 [INFO] - NumExpr defaulting to 8 threads.
2025-07-02 19:38:55,169 [INFO] - Age filtered: 32550 patients, 2080959 records
2025-07-02 19:39:02,862 [INFO] - Time window filtered: 22591 patients, (542184, 104) time-series
2025-07-02 19:39:45,935 [INFO] - Demographics processed: (22591, 4)
2025-07-02 19:39:45,936 [INFO] - Demographic encoders: ['gender_encoded', 'ethnicity_encoded', 'insurance_encoded']
2025-07-02 19:39:45,958 [INFO] - Splits: train=14825, val=2118, test=5648 subjects
2025-07-02 19:39:46,343 [INFO] - Engineering features for 6 categories
2025-07-02 19:45:47,897 [INFO] - Features engineered: (14825, 454)
2025-07-02 19:45:48,066 [INFO] - Engineering features for 6 categories
2025-07-02 19:46:39,368 [INFO] - Features engineered: (2118, 454)
2025-07-02 19:46:39,534 [INFO] - Engineering features for 6 categories
2025-07-02 19:48:56,949 [INFO] - Features engineered: (5648, 454)
2025-07-02 19:48:57,011 [INFO]


✓ Clean data preprocessing completed in 10.53 minutes


## 3. XGBoost Model Training & Evaluation

Run XGBoost analysis with hyperparameter optimization using the preprocessed data:


In [3]:
print("=" * 70)
print("PHASE 2: XGBOOST MODEL TRAINING & EVALUATION")
print("=" * 70)

xgboost_start = time.time()

# Initialize variables for results capture
auroc_results = None
auprc_results = None
final_model = None
xgboost_results = None

try:
    # Import and run XGBoost analysis script  
    import importlib.util
    spec = importlib.util.spec_from_file_location("xgboost_analysis", "xgboost_analysis.py")
    if spec is None:
        raise ImportError("Could not load xgboost_analysis.py")
    
    xgboost_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for xgboost_analysis.py")
    
    print(f"Starting XGBoost analysis at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Execute the XGBoost analysis module and call main function
    spec.loader.exec_module(xgboost_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED,
        'N_OPTUNA_TRIALS': N_OPTUNA_TRIALS,
        'OPTUNA_TIMEOUT': OPTUNA_TIMEOUT
    }
    
    # Call the main function explicitly with configuration
    xgboost_module.main(config)
    
    # Load the results that were saved by the XGBoost script
    results_path = os.path.join(OUTPUT_DIR, 'results_xgboost_baseline.pkl')
    if os.path.exists(results_path):
        with open(results_path, 'rb') as f:
            xgboost_results = pickle.load(f)
        
        # Extract key metrics for display
        auroc_results = {
            'point_estimate': xgboost_results['test_auroc'],
            'ci_lower': xgboost_results['test_auroc_ci_lower'],
            'ci_upper': xgboost_results['test_auroc_ci_upper']
        }
        
        auprc_results = {
            'point_estimate': xgboost_results['test_auprc'], 
            'ci_lower': xgboost_results['test_auprc_ci_lower'],
            'ci_upper': xgboost_results['test_auprc_ci_upper']
        }
        
        print(f"\n✅ XGBoost Results Successfully Captured:")
        print(f"   AUROC: {auroc_results['point_estimate']:.4f} "
              f"(95% CI: {auroc_results['ci_lower']:.4f}-{auroc_results['ci_upper']:.4f})")
        print(f"   AUPRC: {auprc_results['point_estimate']:.4f} "
              f"(95% CI: {auprc_results['ci_lower']:.4f}-{auprc_results['ci_upper']:.4f})")
    else:
        print("⚠️  Results file not found - metrics may not be available in summary")
    
    xgboost_time = time.time() - xgboost_start
    print(f"\n✓ XGBoost analysis completed in {xgboost_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during XGBoost analysis: {e}")
    print(f"   This may be due to XGBoost library installation issues")
    print(f"   Preprocessing completed successfully - data is ready for manual analysis")
    xgboost_time = time.time() - xgboost_start
    print(f"   Attempted runtime: {xgboost_time/60:.2f} minutes")
    # Don't raise - allow pipeline to continue and show preprocessing results


PHASE 2: XGBOOST MODEL TRAINING & EVALUATION
Starting XGBoost analysis at 2025-07-02 19:49:00


/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-02 19:49:01,043 [INFO] - Loading preprocessed data with prefix: preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42
2025-07-02 19:49:01,075 [INFO] - Data shapes: X_train=(14825, 458), X_val=(2118, 458), X_test=(5648, 458)
2025-07-02 19:49:01,075 [INFO] - Using preprocessed data directly - no additional cleaning required
2025-07-02 19:49:01,076 [INFO] - Creating new Optuna study with 15 trials...
[I 2025-07-02 19:49:01,076] A new study created in memory with name: no-name-c325d3c2-34e3-4b57-9dfb-1150ebfa694b
/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/xgboost/sklearn.py:797: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-lea


Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.98      0.96      5077
           1       0.68      0.42      0.52       571

    accuracy                           0.92      5648
   macro avg       0.81      0.70      0.74      5648
weighted avg       0.91      0.92      0.91      5648


Confusion Matrix:
[[4968  109]
 [ 334  237]]

✅ XGBoost Results Successfully Captured:
   AUROC: 0.9080 (95% CI: 0.8968-0.9193)
   AUPRC: 0.6193 (95% CI: 0.5850-0.6559)

✓ XGBoost analysis completed in 8.23 minutes


## 4. Elastic Net Model Training & Evaluation

Run Elastic Net analysis with hyperparameter optimization using the preprocessed data. Note that Elastic Net requires imputation of missing values since it cannot handle NaN values like XGBoost can:


In [4]:
print("=" * 70)
print("PHASE 3: ELASTIC NET MODEL TRAINING & EVALUATION")
print("=" * 70)

elastic_net_start = time.time()

try:
    # Import and run Elastic Net analysis script  
    import importlib.util
    spec = importlib.util.spec_from_file_location("elastic_net_analysis", "elastic_net_analysis.py")
    if spec is None:
        raise ImportError("Could not load elastic_net_analysis.py")
    
    elastic_net_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for elastic_net_analysis.py")
    
    print(f"Starting Elastic Net analysis at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Execute the Elastic Net analysis module
    spec.loader.exec_module(elastic_net_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED,
        'N_OPTUNA_TRIALS': N_OPTUNA_TRIALS,
        'OPTUNA_TIMEOUT': OPTUNA_TIMEOUT
    }
    
    # Call the main function
    elastic_net_module.main(config)
    
    elastic_net_time = time.time() - elastic_net_start
    print(f"\\n✓ Elastic Net analysis completed in {elastic_net_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during Elastic Net analysis: {e}")
    elastic_net_time = time.time() - elastic_net_start
    print(f"   Attempted runtime: {elastic_net_time/60:.2f} minutes")


PHASE 3: ELASTIC NET MODEL TRAINING & EVALUATION
Starting Elastic Net analysis at 2025-07-02 19:57:14


2025-07-02 19:57:15,812 [INFO] - Loading preprocessed data with prefix: preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42
2025-07-02 19:57:15,891 [INFO] - Data shapes: X_train=(14825, 458), X_val=(2118, 458), X_test=(5648, 458)
2025-07-02 19:57:15,892 [INFO] - Using preprocessed data directly - no additional cleaning required
2025-07-02 19:57:15,893 [INFO] - Creating new Optuna study with 15 trials...
[I 2025-07-02 19:57:15,893] A new study created in memory with name: no-name-055b6acd-8f92-4486-82ad-ea82449fa879
[I 2025-07-02 19:58:07,762] Trial 0 finished with value: 0.8608414160056547 and parameters: {'C': 0.09749574450617889, 'l1_ratio': 0.9428998889180539}. Best is trial 0 with value: 0.8608414160056547.
[I 2025-07-02 19:58:08,838] Trial 1 finished with value: 0.6957757402026231 and parameters: {'C': 0.000214189760372204, 'l1_ratio': 0.5297296577304973}. Best is trial 0 with value: 0.8608414160056547.
[I 2025-07-02 19:59:04,975] Trial 2 finished with value: 0.858986000942


Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.83      0.89      5077
           1       0.34      0.81      0.48       571

    accuracy                           0.82      5648
   macro avg       0.66      0.82      0.69      5648
weighted avg       0.91      0.82      0.85      5648


Confusion Matrix:
[[4193  884]
 [ 109  462]]
\n✓ Elastic Net analysis completed in 14.23 minutes
